# Notebook 00. Cache 224 del banano
Input: salida de EDA Banana Leaf. Primero se parte 80/10/10, luego se reduce cada imagen a 224 x 224 y al final se calculan las estadisticas solo con train.

In [ ]:
# CELDA 1: ubicar las imagenes y partir 80/10/10 ANTES de todo
import os, glob, json, numpy as np, pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

META = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)[0]
ROOT = os.path.dirname(META)
CLASES = ["sigatoka", "cordana", "sanas"]
meta = pd.read_csv(META)[["file_name", "class_name"]].copy()
meta["path"] = [os.path.join(ROOT, "data", c, f) for c, f in zip(meta.class_name, meta.file_name)]
assert meta.path.map(os.path.exists).all(), "Hay imagenes que no se encuentran"
meta["label"] = meta.class_name.map({c: i for i, c in enumerate(CLASES)})
meta["sample_id"] = meta.class_name + "/" + meta.file_name
assert meta.sample_id.is_unique

tr, tmp = train_test_split(meta, test_size=0.20, stratify=meta.class_name, random_state=42)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp.class_name, random_state=42)
meta["split"] = "train"
meta.loc[va.index, "split"] = "val"
meta.loc[te.index, "split"] = "test"
print(pd.crosstab(meta.class_name, meta.split, margins=True)[["train", "val", "test", "All"]])

In [ ]:
# CELDA 2: reducir CADA imagen completa a 224 x 224 (las 2,354, sin recorte)
SIZE, OUT = 224, "/kaggle/working"
X = np.lib.format.open_memmap(f"{OUT}/X224.npy", mode="w+", dtype=np.uint8, shape=(len(meta), SIZE, SIZE, 3))
for i, p in enumerate(meta.path):
    im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
    X[i] = np.asarray(im.resize((SIZE, SIZE), Image.BOX))
    if i % 300 == 0:
        print(i, meta.sample_id.iat[i])
X.flush()
print("Cache:", X.shape, X.dtype)

In [ ]:
# CELDA 3: estadisticas calculadas SOLO con train (80 %)
tr_idx = np.where(meta.split == "train")[0]
s, ss, n = np.zeros(3), np.zeros(3), 0
for i in tr_idx:
    x = X[i].astype(np.float64) / 255.0
    s += x.sum((0, 1)); ss += (x ** 2).sum((0, 1)); n += x.shape[0] * x.shape[1]
mean = s / n
std = np.sqrt(ss / n - mean ** 2)
cnt = meta.label.values[tr_idx]
n_c = np.bincount(cnt, minlength=3)
pesos = len(tr_idx) / (3 * n_c)
stats = {"mean_train": mean.tolist(), "std_train": std.tolist(), "imagenes_train_por_clase": n_c.tolist(),
         "pesos_clase_train": pesos.tolist(), "clases": CLASES, "split": "80/10/10 estratificado, semilla 42"}
json.dump(stats, open(f"{OUT}/stats.json", "w"), indent=1)
meta.drop(columns="path").to_csv(f"{OUT}/meta.csv", index=False)
print(json.dumps(stats, indent=1))

In [ ]:
# CELDA 4: verificaciones finales
X = np.load(f"{OUT}/X224.npy", mmap_mode="r")
m = pd.read_csv(f"{OUT}/meta.csv")
assert X.shape == (len(m), 224, 224, 3) and len(m) == 2354
assert m.split.value_counts().to_dict() == {"train": 1883, "val": 235, "test": 236}
assert set(m.sample_id[m.split == "train"]).isdisjoint(m.sample_id[m.split != "train"])
print("CACHE OK:", X.shape, "| train/val/test:", m.split.value_counts().to_dict())
!ls -la /kaggle/working